| Component  | Model                  | Job                         |
| ---------- | ---------------------- | --------------------------- |
| Embedding  | `gemini-embedding-001` | Text → vectors              |
| Generation | `gemini-2.5-flash`     | Context + question → answer |


#  CHUNKING

In [7]:

chunk_size = 500
chunk_overlap = 50

chunks = []

start = 0

while start < len(document):

    end = start + chunk_size

    chunk = document[start:end]

    chunks.append(chunk)

    start = end - chunk_overlap


print("\nTotal chunks:", len(chunks))



Total chunks: 16



# CREATE GEMINI EMBEDDINGS


In [ ]:
Chunk
   ↓
Embedding Model
   ↓
[0.12, -0.45, 0.77, ...]

In [9]:

result = client.models.embed_content( model="gemini-embedding-001", contents=chunks )

chunk_embeddings = np.array(
    [embedding.values for embedding in result.embeddings],
    dtype="float32"
)

print("\nEmbedding shape:", chunk_embeddings.shape)


Embedding shape: (16, 3072)


In [ ]:
16   → number of chunks
3072 → dimensions of each embedding


#  CREATE FAISS INDEX


### Why IndexFlatL2 is good for your student RAG project

For a simple RAG pipeline, it is a very good choice because it is:

    * Easy to understand
    * Easy to implement
    * Exact search
   

### L2 means Euclidean distance.

In [10]:

dimension = chunk_embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(chunk_embeddings)

print("Vectors stored in FAISS:", index.ntotal)

Vectors stored in FAISS: 16


# FAISS Index Mechanisms

| Index           | Search method             | Accuracy  | Speed                    | Best for              |
| --------------- | ------------------------- | --------- | ------------------------ | --------------------- |
| `IndexFlatL2`   | Exact brute-force         | Very high | Slower for huge datasets | Small/medium datasets |
| `IndexFlatIP`   | Exact dot-product         | Very high | Slower for huge datasets | Similarity search     |
| `IndexIVFFlat`  | Approximate/inverted-file | High      | Faster                   | Large datasets        |
| `IndexHNSWFlat` | Graph-based               | High      | Very fast                | Large-scale search    |
